# Enrich Caldova customer feedback

This notebook reads the Fabric SQL shortcut tables in the attached Lakehouse, classifies each feedback record by issue category, assigns a review priority, validates the result, and writes a managed Delta table for ontology binding.

In [ ]:
import synapse.ml.spark.aifunc as aifunc
from pyspark.sql import functions as F

# Include the documented Fabric Runtime 2.0 compatibility settings.
aifunc.default_conf.set_deployment_name("gpt-5-mini")
aifunc.default_conf.set_reasoning_effort("low")
aifunc.default_conf.reset_temperature()

ISSUE_CATEGORIES = [
    "delivery",
    "packaging",
    "instructions",
    "support",
    "scheduling",
    "other",
]
SUPPORTED_SENTIMENTS = ["positive", "negative", "neutral", "mixed"]

In [ ]:
product = spark.table("dbo.product")
feedback = spark.table("dbo.customer_feedback")

required_product_ids = ["P101", "P104", "P113"]
available_product_ids = {row.ProductID for row in product.select("ProductID").collect()}
missing_product_ids = sorted(set(required_product_ids) - available_product_ids)
if missing_product_ids:
    raise RuntimeError(f"Missing required Caldova products: {missing_product_ids}")
if feedback.count() != 16:
    raise RuntimeError("Expected 16 feedback rows from the Fabric SQL shortcut.")
if feedback.filter(F.col("Sentiment").isNull()).count():
    raise RuntimeError("Every feedback row must have sentiment before enrichment.")
if feedback.filter(~F.col("Sentiment").isin(SUPPORTED_SENTIMENTS)).count():
    raise RuntimeError("The source contains an unsupported sentiment label.")
if feedback.join(product, "ProductID", "left_anti").count():
    raise RuntimeError("One or more feedback rows don't resolve to a product.")

In [ ]:
analysis_input = (feedback
    .join(product, "ProductID", "inner")
    .withColumn(
        "AnalysisText",
        F.concat_ws(
            " ",
            F.concat(F.lit("Product: "), F.col("ProductName")),
            F.concat(F.lit("Therapy area: "), F.col("TherapyArea")),
            F.concat(F.lit("Lifecycle stage: "), F.col("LifecycleStage")),
            F.concat(F.lit("Commercial region: "), F.col("CommercialRegion")),
            F.concat(F.lit("Manufacturing type: "), F.col("ManufacturingType")),
            F.concat(F.lit("Customer feedback: "), F.col("FeedbackText")),
        ),
    ))

# This code uses AI. Always review output for mistakes.
classified = analysis_input.ai.classify(
    labels=ISSUE_CATEGORIES,
    input_col="AnalysisText",
    output_col="IssueCategory",
    error_col="CategoryAIError",
)

enriched = (classified
    .withColumn("IssueCategory", F.lower(F.col("IssueCategory")))
    .withColumn(
        "ReviewPriority",
        F.when(F.col("Sentiment") == "negative", F.lit("high"))
         .when(F.col("Sentiment") == "mixed", F.lit("medium"))
         .otherwise(F.lit("standard")),
    ).cache())

In [ ]:
failed = enriched.filter(
    F.col("IssueCategory").isNull()
    | (~F.col("IssueCategory").isin(ISSUE_CATEGORIES))
    | F.col("CategoryAIError").isNotNull()
)
if failed.count():
    display(failed.select("FeedbackID", "IssueCategory", "CategoryAIError"))
    raise RuntimeError("One or more feedback records weren't enriched.")

output_columns = [
    "FeedbackID",
    "ProductID",
    "FeedbackDate",
    "Channel",
    "Region",
    "FeedbackText",
    "Sentiment",
    "IssueCategory",
    "ReviewPriority",
    "CategoryAIError",
]
(enriched.select(*output_columns)
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("dbo.customer_feedback_enriched"))

print("Validation passed: 16 feedback records enriched and written to customer_feedback_enriched.")
display(
    enriched.groupBy("IssueCategory", "ReviewPriority")
            .count()
            .orderBy("IssueCategory", "ReviewPriority")
)
enriched.unpersist()